In [1]:
import glob, os, shutil
src = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/VERSION", recursive=True)
       if open(p).read().strip().startswith("v6")]
assert src, "v6 code not found: attach the dataset with unlearning_tofu_experiment_v6.zip"
dst = "/kaggle/working/unlearning-project"
shutil.rmtree(dst, ignore_errors=True); shutil.copytree(src[0], dst); os.chdir(dst)
!pip install -q rouge-score bitsandbytes
os.makedirs("results/v6", exist_ok=True)
!python env_capture.py > results/v6/environment_run13a.txt
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-53db03ba-5a0e-ef27-c040-762e540d54ee)
GPU 1: Tesla T4 (UUID: GPU-425bfee6-0419-56a9-d82e-07be36521132)


In [2]:
# results of run 11a and run 11b (attached as inputs): the selection rule reads them
import glob, os, shutil, zipfile
os.chdir("/kaggle/working/unlearning-project")
for z in glob.glob("/kaggle/input/**/results_v6_run11*.zip", recursive=True):
    with zipfile.ZipFile(z) as f:
        f.extractall(".", [n for n in f.namelist() if n.startswith("results/v6/run11")])
for p in glob.glob("/kaggle/input/**/run11[ab]/O_s42_raw.json", recursive=True):   # if Kaggle unpacked them
    d = os.path.dirname(p); shutil.copytree(d, f"results/v6/{os.path.basename(d)}", dirs_exist_ok=True)
print(sorted(glob.glob("results/v6/run11*")))
!python experiments_v6.py pick

['results/v6/run11a', 'results/v6/run11b']
[15:40:26]   candidate anchor lr=5e-06 ep10: FQ=0.00229 PQ=0.0783 PrivLeak=-0.05 util=0.96
[15:40:26]   candidate anchor lr=5e-06 ep10: FQ=0.00229 PQ=0.0783 PrivLeak=-0.05 util=0.96
[15:40:26]   candidate anchor_shift lr=1e-05 ep5: FQ=0.0196 PQ=0.00135 PrivLeak=-0.20 util=0.93
[15:40:26]   candidate anchor_shift lr=1e-05 ep5: FQ=0.0196 PQ=0.00135 PrivLeak=-0.20 util=0.93
[15:40:26]   candidate match_off lr=1e-05 ep5: FQ=0.00229 PQ=0.000775 PrivLeak=-0.12 util=0.93
[15:40:26]   candidate match_off lr=1e-05 ep5: FQ=0.00229 PQ=0.000775 PrivLeak=-0.12 util=0.93
[('anchor', 5e-06, 10, 'selection rule on run 11'), ('anchor_shift', 1e-05, 5, 'selection rule on run 11'), ('anchor', 1e-05, 10, 'post hoc (added after run 11)')]
[15:40:26] FINISHED - all steps OK  (0.00 h)


In [3]:
%cd /kaggle/working/unlearning-project
!MODEL3=unsloth/Llama-3.2-1B-Instruct CKPT_DIR=/tmp/ckpt python experiments_v6.py run13a

/kaggle/working
README.md: 100%|███████████████████████████| 4.14k/4.14k [00:00<00:00, 8.45MB/s]
full.json: 100%|███████████████████████████| 1.08M/1.08M [00:00<00:00, 88.9MB/s]
Generating train split: 100%|█████| 4000/4000 [00:00<00:00, 86601.47 examples/s]
full                          4000 rows  columns=['question', 'answer']
retain90.json: 100%|██████████████████████████| 963k/963k [00:00<00:00, 168MB/s]
Generating train split: 100%|████| 3600/3600 [00:00<00:00, 594608.74 examples/s]
retain90                      3600 rows  columns=['question', 'answer']
forget10.json: 100%|██████████████████████████| 115k/115k [00:00<00:00, 103MB/s]
Generating train split: 100%|██████| 400/400 [00:00<00:00, 151995.07 examples/s]
forget10                       400 rows  columns=['question', 'answer']
forget10_perturbed.json: 100%|████████████████| 675k/675k [00:00<00:00, 175MB/s]
Generating train split: 100%|███████| 400/400 [00:00<00:00, 72575.23 examples/s]
forget10_perturbed             400 rows

In [4]:
%cd /kaggle/working/unlearning-project
!zip -qr /kaggle/working/results_v6_run13a.zip results/v6/run13a results/v6/environment_run13a.txt logs/v6

/kaggle/working
